In [63]:
import numpy as np
from src.ai_lib import layers
from src.ai_lib import Model, metrics, SequenceDataLoader
from src.ai_lib import optimizers
from src.ai_lib import losses
from src.ai_lib import preprocessing

import re
import unicodedata

raw_text = """
Maître Corbeau, sur un arbre perché,
Tenait en son bec un fromage.
Maître Renard, par l'odeur alléché,
Lui tint à peu près ce langage :
Et bonjour, Monsieur du Corbeau.
Que vous êtes joli ! que vous me semblez beau !
Sans mentir, si votre ramage
Se rapporte à votre plumage,
Vous êtes le Phénix des hôtes de ces bois.
À ces mots le Corbeau ne se sent pas de joie :
Et pour montrer sa belle voix,
Il ouvre un large bec, laisse tomber sa proie.
Le Renard s'en saisit, et dit : Mon bon Monsieur,
Apprenez que tout flatteur
Vit aux dépens de celui qui l'écoute.
Cette leçon vaut bien un fromage sans doute.
Le Corbeau honteux et confus
Jura, mais un peu tard, qu'on ne l'y prendrait plus.
"""

# Diminishing vocab size 
text = raw_text.lower()
text = text.replace(';', ',')
text = text.replace(':', '.')
text = text.replace('-', ' ') 

# Removing all accents and things like that. English would have been easier
text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode('utf-8')

# We keep the basic alphabet and ponctuation
text = re.sub(r'[^a-z \n.,!?\']', '', text)

print(f"Corpus length : {len(text)}")

tokenizer = preprocessing.CharTokenizer(text)
data_tensor = np.array(tokenizer.encode(text), dtype=np.int32)


print(text)

Corpus length : 686

maitre corbeau, sur un arbre perche,
tenait en son bec un fromage.
maitre renard, par l'odeur alleche,
lui tint a peu pres ce langage .
et bonjour, monsieur du corbeau.
que vous etes joli ! que vous me semblez beau !
sans mentir, si votre ramage
se rapporte a votre plumage,
vous etes le phenix des hotes de ces bois.
a ces mots le corbeau ne se sent pas de joie .
et pour montrer sa belle voix,
il ouvre un large bec, laisse tomber sa proie.
le renard s'en saisit, et dit . mon bon monsieur,
apprenez que tout flatteur
vit aux depens de celui qui l'ecoute.
cette lecon vaut bien un fromage sans doute.
le corbeau honteux et confus
jura, mais un peu tard, qu'on ne l'y prendrait plus.



In [64]:
train_test_split = 0.9

n = int(train_test_split * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

batch_size = 32
block_size = 64  # Numbers of characters to look back at
steps_per_epoch = 100 

# Creating dataloaders
train_loader = SequenceDataLoader(
    data=train_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=steps_per_epoch
)

val_loader = SequenceDataLoader(
    data=val_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=20 # Less steps for validation
)

In [65]:
d_model = 16
n_heads = 2
d_ff = 64

dropout_rate = 0.1

vocab_size = tokenizer.vocab_size

llm_network = layers.Sequential([
    layers.Embedding(vocab_size=vocab_size, d_model=d_model),
    layers.SineCosEncoder(d_model=d_model, max_len=1000),

    layers.Dropout(dropout_rate=dropout_rate),

    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, is_causal=True),
    layers.Dropout(dropout_rate=dropout_rate),
    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, is_causal=True),


    layers.LayerNormalization(n_features=d_model),

    layers.Dropout(dropout_rate=dropout_rate),

    layers.Linear(d_model, n_out=vocab_size)
])

model = Model(llm_network)

In [109]:
learning_rate = 5e-3
weight_decay = 1e-3

optimizer = optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
loss = losses.SoftmaxCrossEntropy()


In [110]:
print("Beginning of training of microgpt")
model.fit(
    dataloader=train_loader,
    epochs=100,
    loss=loss,
    optimizer=optimizer,
    validation_dataloader=val_loader,
    verbose=True,
    metrics=[metrics.accuracy]
)

Beginning of training of microgpt

--- Epoch : 0 ---
Training loss : 0.81897
  accuracy (train) : 0.73136
Validation loss : 5.43924
  accuracy (val) : 0.19661

--- Epoch : 1 ---
Training loss : 0.80705
  accuracy (train) : 0.73567
Validation loss : 5.30809
  accuracy (val) : 0.22175

--- Epoch : 2 ---
Training loss : 0.80628
  accuracy (train) : 0.73636
Validation loss : 5.25002
  accuracy (val) : 0.21655

--- Epoch : 3 ---
Training loss : 0.80243
  accuracy (train) : 0.73733
Validation loss : 5.06547
  accuracy (val) : 0.19944

--- Epoch : 4 ---
Training loss : 0.80798
  accuracy (train) : 0.73634
Validation loss : 5.06626
  accuracy (val) : 0.26269

--- Epoch : 5 ---
Training loss : 0.80296
  accuracy (train) : 0.73720
Validation loss : 5.07755
  accuracy (val) : 0.23438

--- Epoch : 6 ---
Training loss : 0.79180
  accuracy (train) : 0.74067
Validation loss : 5.19049
  accuracy (val) : 0.22534

--- Epoch : 7 ---
Training loss : 0.79160
  accuracy (train) : 0.74063
Validation loss : 5

In [115]:
def generate_text(model, tokenizer, prompt_text="A ", max_new_tokens=100):
    """
    Generates text
    """
    # Important : Turns off setting
    model.sequential.set_training(False)
    
    # Use of kv_cache for inference
    model.sequential.set_use_cache(True)
    model.sequential.reset_cache()
    
    # Encodes context (initial text / prompt)
    context_ids = tokenizer.encode(prompt_text)
    
    # Prefill
    X_input = np.array([context_ids], dtype=np.int32)
    logits = model.predict(X_input)
    
    # We take the predicted char and add it to the list
    next_char_id = np.argmax(logits[0, -1, :])
    context_ids.append(next_char_id)
    
    # Decode
    for _ in range(max_new_tokens - 1):
        # X_input becomes a simple integer as we use kv_cache
        X_input = np.array([[next_char_id]], dtype=np.int32)
        
        logits = model.predict(X_input)
        
        # Since X_input is of length one, logits has shape (1, 1, Vocab_size)
        next_char_id = np.argmax(logits[0, -1, :])
        context_ids.append(next_char_id)
        
    return tokenizer.decode(context_ids)

generated_text = generate_text(model, tokenizer, prompt_text="jura, mais", max_new_tokens=100)
print(generated_text)

jura, maisi ai ait ait age age pe ait age age age pe ai ai age age ro ro o omro o o o o omro ororomroromrororo
